# 01 · Data Understanding

Following Géron Ch. 2 — *"Get the Data"* and *"Take a Quick Look at the Data Structure"*.

Goals:
1. Load and profile train / test / store files.
2. Inspect shape, dtypes, nulls and the target distribution.
3. Create a **time-based** hold-out split — never a random split for time-series.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while not (ROOT / 'configs').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({'figure.dpi': 120})
pd.set_option('display.float_format', '{:.2f}'.format)

from rossmann_store_sales.data import load_train, load_store, load_test, merge_store
from rossmann_store_sales.config import load_config

cfg = load_config(ROOT / 'configs' / 'project.toml')
print('Config loaded.')

## 1. Load raw files

In [ ]:
train = load_train(cfg)
store = load_store(cfg)
test  = load_test(cfg)

print(f'train : {train.shape}')
print(f'store : {store.shape}')
print(f'test  : {test.shape}')

train.head()

In [ ]:
# Column types and non-null counts — equivalent to housing.info() in the book
train.info()

In [ ]:
# Numerical summary — equivalent to housing.describe() in the book
train.describe()

## 2. Categorical columns — value_counts

In [ ]:
for col in ['state_holiday', 'store_type', 'assortment']:
    target = store if col in store.columns else train
    print(target[col].value_counts(), '\n')

## 3. Missing values

In [ ]:
missing = (
    train.isna().sum()
    .rename('train')
    .to_frame()
    .join(store.isna().sum().rename('store'))
    .query('train > 0 or store > 0')
)
print('Columns with missing values:')
missing

**Key observations (same style as Géron's data audit):**
- `competition_distance`: ~2,600 nulls in `store.csv` — will be imputed with a large value (200 km) to signal "no nearby competitor".
- `promo_interval`: only applies to stores enrolled in the continuous Promo2 scheme.
- Competition open-since columns: imputed with the current date when missing (no information ≈ very recent competitor).

## 4. Target distribution — histograms

In [ ]:
sales = train.query('open == 1 and sales > 0')['sales']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(sales, bins=60, edgecolor='white', linewidth=0.4, color='steelblue')
axes[0].set_title('Sales — raw')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

axes[1].hist(np.log1p(sales), bins=60, edgecolor='white', linewidth=0.4, color='darkorange')
axes[1].set_title('Sales — log1p (training target)')

plt.tight_layout()
(ROOT / 'reports/figures').mkdir(parents=True, exist_ok=True)
plt.savefig(ROOT / 'reports/figures/sales_distribution.png', bbox_inches='tight')
plt.show()
print('Skewness | raw:', round(sales.skew(), 2), '| log1p:', round(np.log1p(sales).skew(), 2))

The raw target is **right-skewed** (skewness > 1).  
Applying `log1p` brings it close to a normal distribution — standard practice for gradient-boosted regressors on sales data and noted in the book's data-preparation section.

## 5. Time-based train / validation split

> *"Before you look at the data any further, you need to create a test set, put it aside, and never look at it."* — Géron Ch. 2

For time-series we **always** split on a time boundary to prevent data leakage.  
The last **6 weeks** of the training set become the validation split.

In [ ]:
full = merge_store(train, store)
full['date'] = pd.to_datetime(full['date'])

cutoff   = full['date'].max() - pd.Timedelta(weeks=6)
train_df = full[full['date'] <  cutoff]
valid_df = full[full['date'] >= cutoff]

print(f'Training   : {train_df.shape[0]:>9,} rows  ({train_df["date"].min().date()} → {train_df["date"].max().date()})')
print(f'Validation : {valid_df.shape[0]:>9,} rows  ({valid_df["date"].min().date()} → {valid_df["date"].max().date()})')
print(f'Cut-off    : {cutoff.date()}')

## 6. Save data profile to reports/

In [ ]:
from rossmann_store_sales.data import profile

summary = profile(ROOT / 'configs' / 'project.toml')
print('Data profile written to reports/data_profile.json')
for k, v in summary.items():
    if k != 'missing_by_column':
        print(f'  {k}: {v}')